# Notebook 02 — Data Cleaning & SQLite Loading
**Bluestock Fintech Mutual Fund Analytics Capstone**

In [1]:

import subprocess, sys
result = subprocess.run([sys.executable, '../scripts/etl_pipeline.py'],
                       capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:500])


fund_master:              40 rows
nav_history:           64320 rows  (after ffill)
aum_by_fund_house:        90 rows
monthly_sip_inflows:      48 rows
category_inflows:        144 rows
industry_folio_count:     21 rows
scheme_performance:       40 rows
investor_transactions: 32778 rows
portfolio_holdings:      322 rows
benchmark_indices:      8050 rows
dim_date:               1461 rows

ETL pipeline complete. DB at: /home/claude/mf_analytics/notebooks/../data/db/bluestock_mf.db



## Verify SQLite DB Row Counts

In [2]:

import sqlite3, pandas as pd
from pathlib import Path

DB = Path().resolve().parent / 'data' / 'db' / 'bluestock_mf.db'
conn = sqlite3.connect(DB)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f'Tables in DB: {tables.name.tolist()}\n')
for t in tables['name']:
    cnt = pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn).iloc[0,0]
    print(f'  {t:<30} {cnt:>7} rows')
conn.close()


Tables in DB: ['dim_fund', 'fact_nav', 'fact_aum', 'fact_sip_inflows', 'fact_category_inflows', 'fact_folio_count', 'fact_performance', 'fact_transactions', 'fact_portfolio', 'fact_benchmarks', 'dim_date']

  dim_fund                            40 rows
  fact_nav                         64320 rows
  fact_aum                            90 rows
  fact_sip_inflows                    48 rows
  fact_category_inflows              144 rows
  fact_folio_count                    21 rows
  fact_performance                    40 rows
  fact_transactions                32778 rows
  fact_portfolio                     322 rows
  fact_benchmarks                   8050 rows
  dim_date                          1461 rows


## Run 10 Analytical Queries

In [3]:

import sqlite3, pandas as pd
from pathlib import Path

DB   = Path().resolve().parent / 'data' / 'db' / 'bluestock_mf.db'
conn = sqlite3.connect(DB)

queries = {
    'Top 5 Funds by AUM':
        'SELECT scheme_name, aum_crore FROM fact_performance ORDER BY aum_crore DESC LIMIT 5',
    'Avg NAV per Month (sample)':
        "SELECT SUBSTR(date,1,7) mo, ROUND(AVG(nav),2) avg_nav FROM fact_nav GROUP BY mo LIMIT 10",
    'Top 10 States by Investment':
        "SELECT state, ROUND(SUM(amount_inr)/1e7,2) crore FROM fact_transactions GROUP BY state ORDER BY crore DESC LIMIT 10",
    'Funds with Expense < 1%':
        "SELECT scheme_name, expense_ratio_pct FROM fact_performance WHERE expense_ratio_pct < 1 ORDER BY expense_ratio_pct",
    'T30 vs B30 Summary':
        "SELECT city_tier, COUNT(*) txns, ROUND(SUM(amount_inr)/1e7,2) crore FROM fact_transactions GROUP BY city_tier",
}

for title, q in queries.items():
    print(f'\n--- {title} ---')
    print(pd.read_sql(q, conn).to_string(index=False))

conn.close()



--- Top 5 Funds by AUM ---
                                          scheme_name  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth      49046
        Kotak Emerging Equity Fund - Regular - Growth      47469
       Nippon India Small Cap Fund - Regular - Growth      43630
           DSP Top 100 Equity Fund - Regular - Growth      41828
                  UTI Mid Cap Fund - Regular - Growth      41728

--- Avg NAV per Month (sample) ---
     mo  avg_nav
2022-01   207.05
2022-02   207.72
2022-03   209.69
2022-04   211.79
2022-05   212.73
2022-06   213.86
2022-07   213.95
2022-08   215.70
2022-09   218.47
2022-10   219.53

--- Top 10 States by Investment ---
         state  crore
        Punjab  31.58
    Tamil Nadu  31.52
Madhya Pradesh  30.83
     Rajasthan  29.86
       Gujarat  29.84
   West Bengal  29.72
     Telangana  29.02
         Delhi  28.96
 Uttar Pradesh  28.54
       Haryana  27.96

--- Funds with Expense < 1% ---
                                         scheme